<center><img src="./img/pong-thumbnail.png" width="200" alt="Skills Network Logo"  /></center>
  


## Test Contenedores levantados:


## Dockers:

Para asegurarnos de que todos los servicios levantados en los contenedores funcionan correctamente, podemos definir algunos test automáticos y manuales para verificar cada servicio. A continuación están los pasos y comandos que hemos incluido para testear los contenedores y garantizar que todo está funcionando como se espera:

- ## 1. Verificar el estado de los contenedores
Después de ejecutar `make`, puedos usar este comando para verificar que todos los contenedores estén en ejecución y sin errores:

In [ ]:
make ps
docker compose -f ./src/docker-compose.yml ps
docker ps -a

Asegurarse de que todos los contenedores estén en estado "Up". Si hay alguno en estado "Exited" o "Restarting", eso indicaría un problema con ese servicio.

- ## 2. Comprobación de los logs:
Puedes revisar los logs de cada servicio para ver si se están generando errores o advertencias que no deberían estar allí. Esto es útil para depurar rápidamente problemas de configuración o dependencias faltantes:

In [ ]:
make logs
docker compose -f ./src/docker-compose.yml logs

Podemos filtrar los logs por servicio específico:

In [ ]:
make logs_service
Por favor, especifica un servicio. Uso: make logs_service SERVICE=<nombre_del_servicio>
make logs_service SERVICE=<nombre_del_servicio>
docker compose -f ./src/docker-compose.yml logs <service_name>

- ## 3. Tests específicos por servicio
    - ## a. SQLite (Base de datos):

    Una vez que el contenedor esté levantado, podemos ejecutar comandos SQL para verificar si la base de datos funciona correctamente. Por ejemplo, podemos conectarnos al contenedor de SQLite y listar las tablas para asegurarte de que está configurado:
    - **Test:** Verificar que SQLite está en ejecución
    
- **Comandos:** 

In [ ]:
docker exec -it sqlite sh

/var/lib/sqlite # ls
init_db.sh  sqlite.db

sqlite3 sqlite.db
sqlite>

sqlite> .tables
test

Si obtenemos un listado correcto de tablas, la base de datos está funcionando bien.

Esta vez vemos el scrip que crea la base de datos al levnatar el contenedor y una lista llamada 'test' de prueba.

 - ### Test: Verificar persistencia de datos
 
Insertar un dato en la base de datos y reiniciar el contenedor.



In [ ]:
CREATE TABLE test (id INTEGER PRIMARY KEY, name TEXT);
INSERT INTO test (name) VALUES ('test_entry');

Si ya tenemos una tabla llamada test y nuestro SQLite funciona OK, nos debe de lanzar este error:

In [ ]:
Parse error: table test already exists
  CREATE TABLE test (id INTEGER PRIMARY KEY, name TEXT); INSERT INTO test (name)
               ^--- error here

Por lo tanto cambiaremos nuestro comando para que nos cree la tabla si no existe y que inserte un dato:

In [ ]:
CREATE TABLE IF NOT EXISTS test (id INTEGER PRIMARY KEY, name TEXT);
INSERT INTO test (name) VALUES ('test_entry');

Para ver los datos que hemos insertado ejecutamos este comando:

In [ ]:
SELECT * FROM test;

sqlite> SELECT * FROM test;
1|test_entry
sqlite> .exit
/var/lib/sqlite # exit

Tumbamos y volvemos a levantar el contenedor. Realizamos de nuevo los pasos y el dato debe de serguir en la tabla.

Ahora podemos borrar la tabla con:

In [ ]:
DROP TABLE test;

***
***

- ## b. Servicio Backend (Node.js API):

**Test: Verificar que la API está en ejecución**

- Comando:

    ```yaml
    curl http://localhost:3000/
    ```
- Acción: Realizar una petición GET al endpoint base del backend.
- Resultado esperado: La API debería devolver una respuesta válida, como un mensaje de bienvenida o un JSON con información básica.

    ```yaml
    ¡Hola, mundo desde Node.js!% 
    ```

**Test: Conexión con SQLite**  <font color="green">**(ERROR - Solucionado)**</font>

- Comando:

    ```yaml
    curl http://localhost:3000/api/test_db
    ```

- Nos devuelve un ERROR:
    ```yaml
    <!DOCTYPE html>
    <html lang="en">
    <head>
    <meta charset="utf-8">
    <title>Error</title>
    </head>
    <body>
    <pre>Cannot GET /api/test_db</pre>
    </body>
    </html>
    ```
- Acción: Endpoint que realice una consulta simple a SQLite, como obtener una lista de datos.
- Resultado esperado: Deberías recibir un JSON con los datos almacenados en la base de datos SQLite.

**Test: Verificar persistencia de datos del backend**

- Comando: Insertar un dato en el sistema mediante un endpoint POST y luego reiniciar el contenedor.

In [ ]:
curl -X curl -X POST http://localhost:3000/api/create -d '{"username":"test_user", "email":"test@example.com", "password":"123456"}' -H "Content-Type: application/json"

- Verificar con un GET

In [ ]:
curl http://localhost:3000/api/items

- Resultado esperado: El dato debería persistir después de reiniciar.

- Reconstruir el contenedor backend: Después de realizar estos cambios, necesitas reconstruir tu contenedor para aplicar los cambios en el código.

In [ ]:
make logs_service SERVICE=backend

app  | 
app  | > backend@1.0.0 start
app  | > node app.js
app  | 
app  | Servidor escuchando en http://localhost:3000
app  | Base de datos inicializada correctamente

- Probar las rutas:
    - Usar curl para probar las rutas y asegurarnos de que las consultas están funcionando correctamente. Por ejemplo:

- Obtener todos los usuarios:
- Obtener todos los juegos:
- Probar la conexión a la base de datos:

In [ ]:
curl http://localhost:3000/api/users
[]% 

curl http://localhost:3000/api/games
[]%

curl http://localhost:3000/api/test_db

Conexión a la base de datos exitosa%  

***
***

- ## c. Servicio PHP:

**Test: Verificar que PHP está en ejecución** <font color="green">**(ERROR - Solucionado)**</font>

 - Comando:
    ```yaml
    curl http://localhost:8080/
    ```
- Nos devuelve un error:
    ```yaml
    curl: (56) Recv failure: Connection reset by peer
    ```
- Acción: Realizar una petición GET al servidor PHP.
- Resultado esperado: Debería devolver una página PHP o un mensaje de bienvenida si está correctamente configurado.

In [ ]:
curl http://localhost:8080/

**Test: Conexión con Backend** <font color="red">**(POR SOLUCIONAR)**</font>

- Comando: Crear archivo test en PHP que haga una petición al backend (Node.js).
    ```yaml
    <?php
    $response = file_get_contents('http://backend:3000/api/test');
    echo $response;
    ?>
    ```
-  Acción: Acceder a esta página desde el navegador.
- Resultado esperado: El servidor PHP debería devolver el resultado de la API del backend.


In [ ]:
curl http://localhost:3000/api/test

**Test: logs**

- Comando:
    ```yaml
    make logs_service SERVICE=php
    ```
-  Acción: Acceder a los registros de del servidor.
- Resultado esperado: El servidor PHP debería devolver los logs.


In [ ]:
php  | AH00558: apache2: Could not reliably determine the server's fully qualified domain name, using 172.18.0.7. Set the 'ServerName' directive globally to suppress this message
php  | AH00558: apache2: Could not reliably determine the server's fully qualified domain name, using 172.18.0.7. Set the 'ServerName' directive globally to suppress this message
php  | [Tue Feb 11 17:04:47.086036 2025] [mpm_prefork:notice] [pid 1:tid 1] AH00163: Apache/2.4.62 (Debian) PHP/8.1.31 configured -- resuming normal operations
php  | [Tue Feb 11 17:04:47.086178 2025] [core:notice] [pid 1:tid 1] AH00094: Command line: 'apache2 -D FOREGROUND'
php  | 192.168.65.1 - - [11/Feb/2025:17:06:14 +0000] "GET / HTTP/1.1" 200 74853 "-" "curl/8.7.1"
php  | 192.168.65.1 - - [11/Feb/2025:17:06:41 +0000] "GET / HTTP/1.1" 200 23033 "-" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36"
php  | 192.168.65.1 - - [11/Feb/2025:17:06:42 +0000] "GET /favicon.ico HTTP/1.1" 404 489 "http://localhost:8080/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36"
php  | 192.168.65.1 - - [11/Feb/2025:17:11:34 +0000] "GET / HTTP/1.1" 200 74853 "-" "curl/8.7.1"

El mensaje:

In [ ]:
AH00558: apache2: Could not reliably determine the server's fully qualified domain name, using 172.18.0.7. Set the 'ServerName' directive globally to suppress this message

Es una advertencia común en Apache.

No afecta el funcionamiento del servidor, pero podemos eliminarla configurando la directiva ServerName en la configuración de Apache.

Solución: <font color="red">**(POR SOLUCIONAR)**</font>

- ### Test de Seguridad - Servicio PHP: (<font color="red">**(usuario: www-data)**</font>)

**Configuración Dockerfile** <font color="green">**(ERROR - Solucionado)**</font>

 - Dockerfile:

In [ ]:
FROM php:8.1-apache

RUN docker-php-ext-install pdo pdo_mysql mysqli

# Crear el archivo test.php
RUN echo '<?php \
$response = file_get_contents("http://backend:3000/api/test"); \
echo $response; \
?>' > /var/www/html/test.php

COPY . /var/www/html

RUN chown -R www-data:www-data /var/www/html \
    && chmod -R 755 /var/www/html

EXPOSE 80

CMD ["apache2-foreground"]

### Línea problemática:

- Explicación de la línea:
```yaml
RUN chown -R www-data:www-data /var/www/html \ && chmod -R 755 /var/www/html
```

1. **`chown -R www-data:www-data /var/www/html`:**

    - `chown`: Este comando cambia el propietario de archivos o directorios. En este caso, el parámetro -R indica que debe hacerlo de forma recursiva, es decir, aplicarlo también a todos los archivos y subdirectorios dentro de `/var/www/html`.
    - `www-data:www-data`: Esto significa que el usuario propietario y el grupo propietario del directorio `/var/www/html` serán `www-data`, que es el usuario y grupo predeterminado bajo el cual Apache ejecuta sus procesos.
    - Esto garantiza que el servidor web Apache (que se ejecuta como `www-data`) tenga los permisos necesarios para acceder, modificar y servir los archivos del directorio `/var/www/html`.

2. **`chmod -R 755 /var/www/html`**:

    - `chmod`: Cambia los permisos de archivos o directorios.
    - `755`: Esto significa que el propietario (en este caso, `www-data`) tiene permisos de lectura, escritura y ejecución (7), mientras que los demás usuarios y grupos tienen permisos de lectura y ejecución (5).
    - Este comando también es recursivo (`-R`), por lo que todos los archivos y directorios dentro de `/var/www/html` tendrán estos permisos.

### Problemas de Seguridad
**Aunque esta configuración asegura que Apache puede acceder a los archivos, no es la mejor práctica de seguridad por varias razones:**

1.  #### Permisos 755 en todos los archivos:
    - Dar permisos de ejecución (`x`) a archivos y directorios no siempre es necesario o deseable. **En particular, los archivos PHP y otros archivos estáticos no deberían ser ejecutables.**
2. #### Ejecutar con `www-data` sin aislamiento de usuarios:
    - Aunque www-data es un usuario sin privilegios en el contenedor, es mejor crear un usuario específico para tu aplicación, ya que esto mejora el aislamiento y la seguridad.
3. #### Eliminación de archivos como root:
    - Hemos comprobado, que si cambiamos el propietario de los archivos a `www-data` y luego intentamos manipularlos fuera del contenedor, puede que necesitemos permisos de root, lo que es incómodo y puede causar problemas de seguridad. Y en 42 no tenemos permisos root, por lo que no podemos borrar los volumenes de archivos persistentes.

### Alternativa: Crear un usuario específico
En lugar de usar el usuario `www-data` por defecto, creamos un nuevo usuario en el contenedor y le damos los permisos adecuados para mejorar la seguridad.

### Modificación del Dockerfile para mejorar la seguridad:

In [ ]:
FROM php:8.1-apache

# Instalar extensiones PHP
RUN docker-php-ext-install pdo pdo_mysql mysqli

# Crear un nuevo usuario no privilegiado
RUN useradd -m myuser

# Crear el archivo test.php
RUN echo '<?php \
$response = file_get_contents("http://backend:3000/api/test"); \
echo $response; \
?>' > /var/www/html/test.php

# Copiar los archivos de la aplicación
COPY . /var/www/html

# Cambiar la propiedad de los archivos al nuevo usuario y ajustar permisos
RUN chown -R myuser:myuser /var/www/html \
    && find /var/www/html -type d -exec chmod 755 {} \; \
    && find /var/www/html -type f -exec chmod 644 {} \;

# Exponer el puerto 80
EXPOSE 80

# Cambiar al nuevo usuario no privilegiado
USER myuser

# Iniciar Apache
CMD ["apache2-foreground"]

***
***

- ## d. Servicio Frontend (TypeScript + Tailwind):

**Test: Verificar que el frontend está en ejecución** <font color="red">**(ERROR POR SOLUCIONAR)**</font>

 - Comando:
    ```yaml
    curl http://localhost:3001/
    ```
 - Nos devuelve un error:
    ```yaml
    curl: (56) Recv failure: Connection reset by peer
    ```
- Acción: Realizar una petición GET al servidor frontend.
- Resultado esperado: El frontend debería devolver la aplicación web inicial.

In [ ]:
curl http://localhost:3001/

**Test: Comunicación con Backend**

 - Acción: Asegurarse de que la aplicación frontend puede realizar peticiones al backend.
- Resultado esperado: El frontend debe mostrar los datos que provienen de la API de backend (por ejemplo, una lista de ítems).